# 🏥 Continuous Learning Random Forest Pipeline

### How it works
1. Spring Boot collects snapshots every 3 hours
2. When **500 rows** accumulate → auto-exports to `new_data.csv`, clears the table
3. **Run this notebook** — new batch is merged into `master_dataset.csv` and model is retrained
4. `master_dataset.csv` is capped at **10,000 rows** — oldest rows are dropped when limit is hit

### Files
| File | Purpose |
|---|---|
| `new_data.csv` | Latest 500-row batch from Spring Boot |
| `master_dataset.csv` | Rolling window dataset (max 10,000 rows) |
| `rf_equipment_model.pkl` | Trained Random Forest model |
| `equipment_encoder.pkl` | LabelEncoder for equipment names |
| `feature_scaler.pkl` | StandardScaler for features |

## Cell 1 — Install dependencies (run once)

In [ ]:
!pip install pandas numpy scikit-learn joblib

## Cell 2 — Train / Update model (run after each new_data.csv arrives)

In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor
import joblib


MASTER_DATASET = r"d:\codesirijan\master_dataset.csv"
MODEL_FILE     = r"d:\codesirijan\rf_equipment_model.pkl"
ENCODER_FILE   = r"d:\codesirijan\equipment_encoder.pkl"
SCALER_FILE    = r"d:\codesirijan\feature_scaler.pkl"
MAX_ROWS       = 10_000   # sliding window cap


# -----------------------------
# LOAD NEW DATA
# -----------------------------

new_data = pd.read_csv(r"d:\codesirijan\new_data.csv")
print(f"New batch loaded: {len(new_data)} rows")


# -----------------------------
# MERGE WITH EXISTING DATA
# -----------------------------

if os.path.exists(MASTER_DATASET):
    old_data = pd.read_csv(MASTER_DATASET)
    df = pd.concat([old_data, new_data], ignore_index=True)
    print(f"Merged: {len(old_data)} existing + {len(new_data)} new = {len(df)} rows")
else:
    df = new_data
    print("No master dataset yet — starting fresh.")


# -----------------------------
# SLIDING WINDOW — CAP AT 10,000
# If merged dataset exceeds MAX_ROWS, drop the oldest rows
# -----------------------------

if len(df) > MAX_ROWS:
    rows_dropped = len(df) - MAX_ROWS
    df = df.iloc[rows_dropped:].reset_index(drop=True)  # keep newest MAX_ROWS rows
    print(f"Sliding window: dropped {rows_dropped} oldest rows → master now has {len(df)} rows")
else:
    print(f"Master dataset size: {len(df)} / {MAX_ROWS} rows")


# Save updated master dataset
df.to_csv(MASTER_DATASET, index=False)
print(f"Master dataset saved → {MASTER_DATASET}")


# -----------------------------
# DROP UNUSED COLUMNS
# -----------------------------

df = df.drop(columns=["recorded_at", "date"], errors="ignore")


# -----------------------------
# REMOVE MISSING VALUES
# -----------------------------

df = df.dropna()


# -----------------------------
# ENCODE EQUIPMENT
# -----------------------------

encoder = LabelEncoder()
df["equipment_encoded"] = encoder.fit_transform(df["equipment"])


# -----------------------------
# BOOLEAN → INTEGER
# -----------------------------

df["is_holiday"] = df["is_holiday"].astype(int)
df["is_weekend"]  = df["is_weekend"].astype(int)


# -----------------------------
# FEATURE SET
# -----------------------------

features = [
    "equipment_encoded",
    "is_holiday",
    "is_weekend",
    "total_patients_last_7_days",
    "total_patients_last_day",
    "total_usage_last_7_days",
    "total_usage_last_day"
]

target = "value"

X = df[features]
y = df[target]


# -----------------------------
# NORMALIZATION
# -----------------------------

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# -----------------------------
# TRAIN RANDOM FOREST
# -----------------------------

model = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    random_state=42,
    n_jobs=-1
)

model.fit(X_scaled, y)


# -----------------------------
# SAVE MODEL
# -----------------------------

joblib.dump(model,   MODEL_FILE)
joblib.dump(encoder, ENCODER_FILE)
joblib.dump(scaler,  SCALER_FILE)

print("Random Forest model updated successfully")
print(f"  Model   → {MODEL_FILE}")
print(f"  Encoder → {ENCODER_FILE}")
print(f"  Scaler  → {SCALER_FILE}")

## Cell 3 — Inference (predict equipment demand)

In [ ]:
import numpy as np
import joblib

model   = joblib.load(r"d:\codesirijan\rf_equipment_model.pkl")
encoder = joblib.load(r"d:\codesirijan\equipment_encoder.pkl")
scaler  = joblib.load(r"d:\codesirijan\feature_scaler.pkl")

# ── Change these values for your prediction ───────────────────────────
equipment = "Ventilators"

input_data = np.array([[
    encoder.transform([equipment])[0],
    0,    # is_holiday             (1 = yes)
    0,    # is_weekend             (1 = yes)
    120,  # total_patients_last_7_days
    18,   # total_patients_last_day
    25,   # total_usage_last_7_days
    4     # total_usage_last_day
]])
# ───────────────────────────────────────────────────────────────────────

input_scaled = scaler.transform(input_data)
prediction   = model.predict(input_scaled)

print("Predicted equipment needed:", round(prediction[0]))